Ноутбук считает retrieval и instruction-following метрики на русском объединенном корпусе. Здесь оцениваются original, instructed, reversed и p-MRR режимы.


In [ ]:
!pip -q install faiss-cpu peft==0.13.2 bitsandbytes==0.46.1 sentencepiece protobuf accelerate

import os, json, math, time, hashlib, gc
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import faiss

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import PeftModel


ENC_DIR = "/kaggle/input/datasets/sukiss/corpus-embedings"  # <-- поменяй на свой Kaggle Dataset

METRIC_TESTSET = "/kaggle/input/datasets/sukiss/dataset-for-metrics/chunks_testset_metric_instructions.jsonl"  # <-- поменяй

BASE_MODEL = "meta-llama/Llama-2-7b-hf"
ADAPTER_DIR = "/kaggle/input/datasets/sukiss/adapters"  # <-- твой LoRA adapter

CORPUS_MODE = "combined"

OUT_DIR = "/kaggle/working/metric_eval"
os.makedirs(OUT_DIR, exist_ok=True)

PER_QUERY_OUT = os.path.join(OUT_DIR, f"metric_eval_{CORPUS_MODE}_per_query.jsonl")
SUMMARY_OUT = os.path.join(OUT_DIR, f"metric_eval_{CORPUS_MODE}_summary.json")

QUERY_MAX_LEN = 192
BATCH_QUERIES = 32

WISE_K = 20
IR_K = 10
MAP_K = 1000

HF_TOKEN = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
if HF_TOKEN:
    login(token=HF_TOKEN) if HF_TOKEN else None


def sha1_text(t: str) -> str:
    return hashlib.sha1(t.strip().encode("utf-8")).hexdigest()

def load_docids(path):
    with open(path, "r", encoding="utf-8") as f:
        return [x.strip() for x in f if x.strip()]

def infer_dim_from_file(emb_path, n_rows, dtype=np.float16):
    bytes_total = os.path.getsize(emb_path)
    bytes_per = np.dtype(dtype).itemsize
    return bytes_total // (n_rows * bytes_per)

def load_one_corpus(enc_dir, name):
    corpus_path = os.path.join(enc_dir, f"{name}_corpus.jsonl")
    docids_path = os.path.join(enc_dir, f"{name}_docids.txt")
    emb_path = os.path.join(enc_dir, f"{name}_embeddings.npy")

    assert os.path.exists(corpus_path), corpus_path
    assert os.path.exists(docids_path), docids_path
    assert os.path.exists(emb_path), emb_path

    docids = load_docids(docids_path)
    dim = infer_dim_from_file(emb_path, len(docids), dtype=np.float16)
    emb = np.memmap(emb_path, dtype=np.float16, mode="r", shape=(len(docids), dim))

    text_hashes = []
    with open(corpus_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            text_hashes.append(sha1_text(obj["text"]))

    assert len(text_hashes) == len(docids)
    return docids, text_hashes, emb, dim

def load_corpus(enc_dir, mode):
    if mode == "test":
        docids, hashes, emb, dim = load_one_corpus(enc_dir, "test")
        xb = np.asarray(emb, dtype=np.float32)
        return docids, hashes, xb, dim

    if mode == "main":
        docids, hashes, emb, dim = load_one_corpus(enc_dir, "main")
        xb = np.asarray(emb, dtype=np.float32)
        return docids, hashes, xb, dim

    if mode == "combined":
        main_docids, main_hashes, main_emb, dim1 = load_one_corpus(enc_dir, "main")
        test_docids, test_hashes, test_emb, dim2 = load_one_corpus(enc_dir, "test")
        assert dim1 == dim2

        docids = [f"main::{x}" for x in main_docids] + [f"test::{x}" for x in test_docids]
        hashes = main_hashes + test_hashes
        xb = np.vstack([
            np.asarray(main_emb, dtype=np.float32),
            np.asarray(test_emb, dtype=np.float32),
        ])
        return docids, hashes, xb, dim1

    raise ValueError(f"Unknown CORPUS_MODE: {mode}")

def build_hash_to_indices(text_hashes):
    m = defaultdict(list)
    for i, h in enumerate(text_hashes):
        m[h].append(i)
    return m

def load_metric_items(path):
    items = []
    bad = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            it = json.loads(line)

            required = [
                "query_id", "query", "only_instruction",
                "reverse_instruction", "pmrr_instruction",
                "positive", "pmrr_changed_docs",
            ]

            if any(not it.get(k) for k in required):
                bad["missing_required"] += 1
                continue

            items.append(it)

    print("Loaded metric items:", len(items))
    print("Skipped:", bad)
    return items

def query_text(item, mode):
    q = item["query"].strip()

    if mode == "orig":
        return q
    if mode == "inst":
        return (q + " " + item["only_instruction"].strip()).strip()
    if mode == "rev":
        return (q + " " + item["reverse_instruction"].strip()).strip()
    if mode == "pmrr":
        return (q + " " + item["pmrr_instruction"].strip()).strip()

    raise ValueError(mode)

def mrr_at_k(rank, k):
    return 1.0 / rank if rank <= k else 0.0

def ndcg_at_k(rank, k):
    return 1.0 / math.log2(rank + 1) if rank <= k else 0.0

def ap_at_k(rank, k):
    return 1.0 / rank if rank <= k else 0.0

def hit_at_k(rank, k):
    return 1.0 if rank <= k else 0.0

def pmrr_doc_score(rank_old, rank_new):
    mrr_old = 1.0 / rank_old
    mrr_new = 1.0 / rank_new

    if rank_old > rank_new:
        return (mrr_old / mrr_new) - 1.0
    return 1.0 - (mrr_new / mrr_old)

def sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev):
    return float(
        (r_ins < r_ori) and
        (s_ins > s_ori) and
        (r_ori < r_rev) and
        (s_ori > s_rev)
    )

def wise_score(r_ori, r_ins, r_rev, n_positive_original=1, k=20):
    if r_ins <= r_ori < r_rev:
        if r_ori <= n_positive_original and r_ins == 1:
            return 1.0
        if r_ori <= k:
            return (1.0 - (math.sqrt(max(0, r_ori - r_ins)) / k)) * (1.0 / math.sqrt(r_ins))
        return 0.01

    if r_rev < r_ori < r_ins:
        return -1.0
    if r_ori <= r_ins:
        return (r_ori - r_ins) / r_ins
    if r_rev <= r_ori:
        return (r_rev - r_ori) / r_ori

    return 0.0

def best_rank_score_from_search(I_row, D_row, candidate_indices):
    candidates = set(candidate_indices)
    for pos, idx in enumerate(I_row):
        if int(idx) in candidates:
            return pos + 1, float(D_row[pos])
    return len(I_row) + 1, float("-inf")

def summarize_mode(rows, prefix):
    n = len(rows)
    ranks = [r[f"rank_{prefix}"] for r in rows]

    return {
        "MRR@10": float(np.mean([mrr_at_k(r, 10) for r in ranks])),
        "nDCG@10": float(np.mean([ndcg_at_k(r, 10) for r in ranks])),
        "MAP@1000": float(np.mean([ap_at_k(r, 1000) for r in ranks])),
        "Hit@10": float(np.mean([hit_at_k(r, 10) for r in ranks])),
        "avg_rank": float(np.mean(ranks)),
        "median_rank": float(np.median(ranks)),
    }


docids, text_hashes, xb, dim = load_corpus(ENC_DIR, CORPUS_MODE)
hash_to_indices = build_hash_to_indices(text_hashes)

print("Corpus mode:", CORPUS_MODE)
print("Docs:", len(docids), "dim:", dim)
print("Embeddings GB:", round(xb.nbytes / 1024**3, 3))

index = faiss.IndexFlatIP(dim)
index.add(xb)
print("FAISS index ready:", index.ntotal)


gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=False, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

base = AutoModel.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN if HF_TOKEN else None,
)
base.config.use_cache = False

model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)
model.eval()

print("Loaded model. device:", next(model.parameters()).device)

def add_eos(texts):
    return [t + tok.eos_token for t in texts]

def eos_pool(last_hidden_state, attention_mask):
    lengths = attention_mask.sum(dim=1) - 1
    idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)
    return last_hidden_state[idx, lengths]

@torch.inference_mode()
def encode_texts(texts, max_len):
    batch = tok(
        add_eos(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    batch = {k: v.to(model.device) for k, v in batch.items()}
    out = model(**batch)
    emb = eos_pool(out.last_hidden_state, batch["attention_mask"])
    emb = F.normalize(emb, p=2, dim=-1)
    return emb.detach().to(torch.float32).cpu().numpy()

def encode_all_queries(texts, batch_size=32):
    out = []
    for i in range(0, len(texts), batch_size):
        out.append(encode_texts(texts[i:i + batch_size], QUERY_MAX_LEN))
    return np.vstack(out)


items = load_metric_items(METRIC_TESTSET)

kept = []
missing_positive = 0
for it in items:
    pos_hash = sha1_text(it["positive"])
    if pos_hash not in hash_to_indices:
        missing_positive += 1
        continue
    kept.append(it)

items = kept
print("Items kept:", len(items))
print("Missing positive in corpus:", missing_positive)

if not items:
    raise ValueError("No items left. Use CORPUS_MODE='test' or 'combined'.")

texts_orig = [query_text(it, "orig") for it in items]
texts_inst = [query_text(it, "inst") for it in items]
texts_rev = [query_text(it, "rev") for it in items]
texts_pmrr = [query_text(it, "pmrr") for it in items]

t0 = time.time()

q_orig = encode_all_queries(texts_orig, BATCH_QUERIES)
q_inst = encode_all_queries(texts_inst, BATCH_QUERIES)
q_rev = encode_all_queries(texts_rev, BATCH_QUERIES)
q_pmrr = encode_all_queries(texts_pmrr, BATCH_QUERIES)

print("Encoded query variants:", q_orig.shape, "time_s:", round(time.time() - t0, 1))

topk = len(docids)

D_orig, I_orig = index.search(q_orig, topk)
D_inst, I_inst = index.search(q_inst, topk)
D_rev, I_rev = index.search(q_rev, topk)
D_pmrr, I_pmrr = index.search(q_pmrr, topk)

rows = []
stats = Counter()

for i, it in enumerate(items):
    qid = str(it["query_id"])
    style = it.get("instruction_style", "unknown")

    pos_hash = sha1_text(it["positive"])
    pos_candidates = hash_to_indices[pos_hash]

    r_ori, s_ori = best_rank_score_from_search(I_orig[i], D_orig[i], pos_candidates)
    r_ins, s_ins = best_rank_score_from_search(I_inst[i], D_inst[i], pos_candidates)
    r_rev, s_rev = best_rank_score_from_search(I_rev[i], D_rev[i], pos_candidates)
    r_pmrr_pos, s_pmrr_pos = best_rank_score_from_search(I_pmrr[i], D_pmrr[i], pos_candidates)

    sicr = sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev)
    wise = wise_score(r_ori, r_ins, r_rev, n_positive_original=1, k=WISE_K)

    changed_scores = []
    changed_details = []

    for ch in it.get("pmrr_changed_docs", []):
        text_hash = ch.get("text_hash") or sha1_text(ch.get("text", ""))
        candidate_indices = hash_to_indices.get(text_hash)

        if not candidate_indices:
            stats["missing_pmrr_changed_doc"] += 1
            continue

        r_old, s_old = best_rank_score_from_search(I_inst[i], D_inst[i], candidate_indices)
        r_new, s_new = best_rank_score_from_search(I_pmrr[i], D_pmrr[i], candidate_indices)

        score = pmrr_doc_score(r_old, r_new)
        changed_scores.append(score)

        changed_details.append({
            "doc_id": ch.get("doc_id"),
            "rank_old_inst": r_old,
            "score_old_inst": s_old,
            "rank_new_pmrr": r_new,
            "score_new_pmrr": s_new,
            "pmrr_doc_score": score,
            "reason": ch.get("reason", ""),
            "text_hash": text_hash,
        })

    pmrr_query = float(np.mean(changed_scores)) if changed_scores else None

    row = {
        "query_id": qid,
        "instruction_style": style,

        "rank_orig": r_ori,
        "score_orig": s_ori,
        "rank_inst": r_ins,
        "score_inst": s_ins,
        "rank_rev": r_rev,
        "score_rev": s_rev,
        "rank_pmrr_positive": r_pmrr_pos,
        "score_pmrr_positive": s_pmrr_pos,

        "mrr10_orig": mrr_at_k(r_ori, IR_K),
        "mrr10_inst": mrr_at_k(r_ins, IR_K),
        "mrr10_rev": mrr_at_k(r_rev, IR_K),

        "ndcg10_orig": ndcg_at_k(r_ori, IR_K),
        "ndcg10_inst": ndcg_at_k(r_ins, IR_K),
        "ndcg10_rev": ndcg_at_k(r_rev, IR_K),

        "map1000_orig": ap_at_k(r_ori, MAP_K),
        "map1000_inst": ap_at_k(r_ins, MAP_K),
        "map1000_rev": ap_at_k(r_rev, MAP_K),

        "hit10_orig": hit_at_k(r_ori, IR_K),
        "hit10_inst": hit_at_k(r_ins, IR_K),
        "hit10_rev": hit_at_k(r_rev, IR_K),

        "sicr": sicr,
        "wise": wise,
        "pmrr": pmrr_query,
        "pmrr_changed_docs_eval": changed_details,
    }

    rows.append(row)

with open(PER_QUERY_OUT, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def summarize_rows(rows):
    pmrr_values = [r["pmrr"] for r in rows if r["pmrr"] is not None]

    summary = {
        "count": len(rows),
        "corpus_mode": CORPUS_MODE,
        "docs": len(docids),

        "orig": summarize_mode(rows, "orig"),
        "inst": summarize_mode(rows, "inst"),
        "rev": summarize_mode(rows, "rev"),

        "delta": {
            "dMRR@10_inst_minus_orig": float(np.mean([r["mrr10_inst"] - r["mrr10_orig"] for r in rows])),
            "dnDCG@10_inst_minus_orig": float(np.mean([r["ndcg10_inst"] - r["ndcg10_orig"] for r in rows])),
            "dMAP@1000_inst_minus_orig": float(np.mean([r["map1000_inst"] - r["map1000_orig"] for r in rows])),
        },

        "robustness": {
            "Robustness@10_min_ndcg_orig_inst_rev": float(np.mean([
                min(r["ndcg10_orig"], r["ndcg10_inst"], r["ndcg10_rev"])
                for r in rows
            ])),
        },

        "instruction_metrics": {
            "SICR": float(np.mean([r["sicr"] for r in rows])),
            "SICR_x100": float(100 * np.mean([r["sicr"] for r in rows])),
            "WISE": float(np.mean([r["wise"] for r in rows])),
            "WISE_x100": float(100 * np.mean([r["wise"] for r in rows])),
            "pMRR": float(np.mean(pmrr_values)) if pmrr_values else None,
            "pMRR_x100": float(100 * np.mean(pmrr_values)) if pmrr_values else None,
            "pMRR_count": len(pmrr_values),
        },

        "avg_gold_rank": {
            "Rori": float(np.mean([r["rank_orig"] for r in rows])),
            "Rins": float(np.mean([r["rank_inst"] for r in rows])),
            "Rrev": float(np.mean([r["rank_rev"] for r in rows])),
        },
    }

    return summary

summary = summarize_rows(rows)

by_style = {}
for style in sorted(set(r["instruction_style"] for r in rows)):
    style_rows = [r for r in rows if r["instruction_style"] == style]
    by_style[style] = summarize_rows(style_rows)

summary["by_style"] = by_style
summary["stats"] = dict(stats)
summary["outputs"] = {
    "per_query": PER_QUERY_OUT,
    "summary": SUMMARY_OUT,
}

with open(SUMMARY_OUT, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDONE")
print("Per-query:", PER_QUERY_OUT)
print("Summary:", SUMMARY_OUT)

print("\nOVERALL:")
print(json.dumps({
    "count": summary["count"],
    "orig": summary["orig"],
    "inst": summary["inst"],
    "rev": summary["rev"],
    "delta": summary["delta"],
    "instruction_metrics": summary["instruction_metrics"],
    "avg_gold_rank": summary["avg_gold_rank"],
    "stats": summary["stats"],
}, ensure_ascii=False, indent=2))

print("\nBY STYLE:")
for style, s in summary["by_style"].items():
    print(style, json.dumps({
        "count": s["count"],
        "SICR_x100": s["instruction_metrics"]["SICR_x100"],
        "WISE_x100": s["instruction_metrics"]["WISE_x100"],
        "pMRR_x100": s["instruction_metrics"]["pMRR_x100"],
        "Rori": s["avg_gold_rank"]["Rori"],
        "Rins": s["avg_gold_rank"]["Rins"],
        "Rrev": s["avg_gold_rank"]["Rrev"],
    }, ensure_ascii=False))
